**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Modern Architectures

The transformer's descendants, dissected: mixture-of-experts routing (capacity without compute), attention's cost curve and its linear/sliding-window repairs, and where [SSM blocks](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) fit. Each mechanism built small and measured.

## 1. Pre-requisites

[Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb), [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb), [Scale_NN](./Scale_NN/Scale_NN.ipynb).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
torch.manual_seed(0)

---
### 🕐 Session 1 of 3 — *Attention's Cost Curve* (~35 min)
**Goal:** measure the quadratic wall; see what sliding windows and linear attention trade away.
**Builds on:** [Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (MoE).

---

## 2. The Quadratic Wall, Measured

💡 **Intuition.** Full attention lets every token query every token: $O(T^2)$ compute and memory — the price of unlimited connectivity. The repairs each *remove* something: **sliding windows** keep only local links (recover long range by stacking layers — the [CNN receptive-field](./Intro_CNN/Intro_CNN.ipynb) trick); **linear attention** replaces softmax with a kernel so the sum factorizes into a running state ($O(T)$ — and mathematically an [SSM](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb)!). No free lunch: each buys speed with a connectivity prior.

In [ ]:

# YOUR CODE HERE


---
### 🕐 Session 2 of 3 — *Mixture of Experts* (~40 min)
**Goal:** route tokens to specialists: parameters without proportional compute — specialization measured.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (the assembled zoo).

---

## 3. Capacity Without the Bill

💡 **Intuition.** An MoE layer holds $E$ expert MLPs but a learned **router** sends each token to only the top-$k$ — so parameters scale with $E$ while per-token compute scales with $k$. The bet: tokens differ in *kind*, and specialists beat one generalist of equal compute. The classic failure is **routing collapse** (all tokens to one expert), patched with load-balancing losses. We build a 4-expert layer on a task with planted sub-populations and *check who goes where*.

In [ ]:
# task with 4 planted regimes: y depends on x differently per quadrant of a latent code
        # load-balancing auxiliary: encourage uniform expert usage
# THE FAILURE, on purpose: no balancing, no exploration → routing collapse
# THE CURE: load-balancing loss + routing noise

# YOUR CODE HERE


---
### 🕐 Session 3 of 3 — *The Assembled Zoo* (~30 min)
**Goal:** how the pieces combine in 2026-era models; the design-space map.
**Builds on:** Session 2.

---

## 4. The Map

| Need | Mechanism | Cost model |
|---|---|---|
| Unlimited connectivity | full attention | $O(T^2)$, KV cache $O(T)$ |
| Long context, cheap | sliding window + a few global layers | $O(Tw)$ |
| Constant-state streaming | linear attention / [SSM/Mamba](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) | $O(T)$, state $O(1)$ |
| Parameters ≫ compute | MoE (top-k routing) | params ×E, FLOPs ×k |
| Memory during training | grouped/multi-query attention, [gradient accumulation](./Scale_NN/Scale_NN.ipynb) | smaller KV, same math |

💡 **Intuition.** Modern frontier models are *hybrids by necessity*: interleaved sliding/full attention, MoE feed-forwards, sometimes SSM layers — each mechanism spending a different currency (compute, memory, connectivity). Read any architecture paper as a walk through this table.

**Exercise with teeth:** wire the MoE layer into the [nano-GPT](./LLMs_from_the_Ground_Up.ipynb) and measure perplexity per FLOP against the dense baseline.

---
## Where next

- [State-Space Models](../Intro_Time_Series/State_Space_Models_Kalman_to_Mamba.ipynb) — the third pillar, in depth.
- [Scale_NN](./Scale_NN/Scale_NN.ipynb) — why these trade-offs exist at all.